In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
import json, pickle

In [2]:
# ── paths ──────────────────────────────────────────────────────────────────────
ROOT = Path('../..').resolve()
ANALYSIS = ROOT / "analysis" / "affective_subspace_coverage"
ACTIVATION = ROOT / "activation" / "emotion_rewrites"
FIGURES = ROOT / "thesis" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── colour / style ─────────────────────────────────────────────────────────────
EMOTION_COLOURS = {
    "joy": "#F4C542",
    "trust": "#5BAD6F",
    "fear": "#7B5EA7",
    "surprise": "#F08030",
    "sadness": "#5B8DB8",
    "disgust": "#8B5E3C",
    "anger": "#D94040",
    "anticipation": "#E07840",
}
CANDIDATE_LAYERS = [10, 13, 16]

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

In [3]:
def plot_layer_lineplot(save_path: Path) -> None:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df.sort_values("layer")

    metrics = [
        ("emo_probe_bacc_pc4",      "Emotion-category probe\nbalanced accuracy (4-D subspace)"),
        ("centroid_evr4",           r"Centroid compactness EVR$_4$"),
        ("mean_local_pc1_int_rho",  "Mean local PC1 –\nintensity Spearman ρ"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)

    for ax, (col, label) in zip(axes, metrics):
        ax.plot(df["layer"], df[col], marker="o", color="#2C5F8A", linewidth=1.8,
                markersize=6, zorder=3)

        # highlight layer 13
        val13 = float(df.loc[df["layer"] == 13, col].iloc[0])
        ax.axvline(13, color="#D94040", linewidth=0.8, linestyle="--", zorder=2,
                   label="Layer 13")
        ax.scatter([13], [val13], color="#D94040", zorder=4, s=50)

        ax.set_xlabel("Layer")
        ax.set_ylabel(label, labelpad=4)
        ax.set_xticks(df["layer"].tolist())
        ax.tick_params(axis="both", which="major", labelsize=9)

    axes[0].legend(fontsize=8, loc="lower right")
    fig.suptitle(
        "Layer-wise diagnostics for affective residual representations",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [4]:
def load_activations():
    """Return H[N, E, I, L, D] and metadata.

    The .npy file may be a raw float32 binary without a numpy header (produced
    by a custom writer).  In that case np.load raises 'invalid load key'.  We
    fall back to np.memmap so the full 50 GB is never paged into RAM; only the
    slices actually accessed are read from disk.
    """
    info = json.loads((ACTIVATION / "emotion_intensity_residual_stream_info.json").read_text())
    npy_path = ACTIVATION / "emotion_intensity_residual_stream.npy"
    try:
        H = np.load(npy_path, allow_pickle=True)
        if isinstance(H, np.ndarray) and H.dtype == object:
            obj = H.item()
            if isinstance(obj, dict):
                H = max(obj.values(), key=lambda v: v.size if hasattr(v, "size") else 0)
            elif isinstance(obj, np.ndarray):
                H = obj
    except Exception:
        # File lacks a numpy header — memory-map as raw binary (no heap allocation)
        shape = tuple(info["shape"])
        dtype = np.dtype(info.get("dtype", "float32"))
        H = np.memmap(npy_path, dtype=dtype, mode="r", shape=shape)
    return H, info


In [5]:
def plot_centroid_pca(save_path: Path) -> None:
    H, info = load_activations()
    layer_indices: list[int] = info["layer_indices"]   # e.g. [8,10,13,16,19,22]
    emotions: list[str]      = info["emotion_order"]
    # H shape: [N, E, I, L, D]

    target_layers = [10, 13, 16]
    layer_pos = {l: layer_indices.index(l) for l in target_layers}

    # mean over N (source texts) and I (intensities) → centroid per emotion per layer
    # H: [N, E, I, L, D]
    centroids = {}
    for l, lp in layer_pos.items():
        # mean over axis 0 (N) and axis 2 (I)
        c = H[:, :, :, lp, :].mean(axis=(0, 2))  # [E, D]
        centroids[l] = c

    # fit a shared PCA on all centroids concatenated
    all_c = np.concatenate([centroids[l] for l in target_layers], axis=0)  # [3E, D]
    pca = PCA(n_components=2)
    pca.fit(all_c)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

    for ax, l in zip(axes, target_layers):
        c_2d = pca.transform(centroids[l])  # [E, 2]
        for i, emo in enumerate(emotions):
            colour = EMOTION_COLOURS.get(emo, "#888888")
            ax.scatter(c_2d[i, 0], c_2d[i, 1], color=colour, s=70, zorder=3)
            ax.annotate(
                emo,
                (c_2d[i, 0], c_2d[i, 1]),
                textcoords="offset points",
                xytext=(5, 3),
                fontsize=7.5,
                color=colour,
            )
        ev1 = pca.explained_variance_ratio_[0] * 100
        ev2 = pca.explained_variance_ratio_[1] * 100
        ax.set_xlabel(f"PC1 ({ev1:.1f}%)", fontsize=9)
        ax.set_ylabel(f"PC2 ({ev2:.1f}%)", fontsize=9)
        ax.set_title(f"Layer {l}", fontsize=10, fontweight="bold")
        ax.axhline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.axvline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        "PCA of emotion centroids (shared basis) at candidate layers",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [6]:
def generate_latex_table() -> str:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df[df["layer"].isin(CANDIDATE_LAYERS)].sort_values("layer")

    rows = []
    best = {
        "emo_probe_bacc_pc4": df["emo_probe_bacc_pc4"].max(),
        "centroid_evr4": df["centroid_evr4"].max(),
        "mean_local_pc1_int_rho": df["mean_local_pc1_int_rho"].max(),
    }

    for _, row in df.iterrows():
        layer = int(row["layer"])
        marker = r" \textbf{*}" if layer == 13 else ""

        def fmt(col, fmt_str):
            v = row[col]
            s = fmt_str.format(v)
            if abs(v - best[col]) < 1e-9:
                s = r"\textbf{" + s + "}"
            return s

        rows.append(
            f"  {layer}{marker} & "
            f"{fmt('emo_probe_bacc_pc4', '{:.4f}')} & "
            f"{fmt('centroid_evr4', '{:.4f}')} & "
            f"{fmt('mean_local_pc1_int_rho', '{:.4f}')} \\\\"
        )

    body = "\n".join(rows)
    table = r"""\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}
\end{table}"""
    return table

In [7]:
plot_layer_lineplot(FIGURES / "layer_diagnostics_lineplot.pdf")

try:
    plot_centroid_pca(FIGURES / "layer_centroid_pca.pdf")
except Exception as e:
    print(f"[WARN] PCA scatter skipped: {e}")

tex = generate_latex_table()
print("\n── LaTeX table ────────────────────────────────────────────────")
print(tex)
out_path = FIGURES / "layer_diagnostics_table.tex"
out_path.write_text(tex)
print(f"\nSaved: {out_path}")

Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/layer_diagnostics_lineplot.pdf
Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/layer_centroid_pca.pdf

── LaTeX table ────────────────────────────────────────────────
\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
  10 & 0.6751 & 0.8876 & 0.0998 \\
  13 \textbf{*} & \textbf{0.7033} & 0.8920 & \textbf{0.1336} \\
  16 & 0.6704 & \textbf{0.8958} & 0.1297 \\
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}


In [8]:

# ── CAA geometry paths ─────────────────────────────────────────────────────────
CAA_GEOMETRY = ROOT / "analysis" / "caa" / "geometry"


In [9]:

def plot_intensity_consistency_heatmap(save_path: Path) -> None:
    """
    Heatmap of per-emotion, per-pair cosine similarity at layer 13.
    Colour range is clipped to [0.980, 1.000] to reveal fine differences.
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_intensity_consistency.csv")
    df13 = df[df["layer"] == 13].copy()

    emotions_order = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
    pairs_order    = ["low-medium", "low-high", "medium-high"]

    mat = (
        df13.pivot(index="emotion", columns="pair", values="cosine")
        .reindex(index=emotions_order, columns=pairs_order)
    )

    vmin, vmax = 0.980, 1.000

    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    im = ax.imshow(mat.values, aspect="auto", cmap="Blues",
                   vmin=vmin, vmax=vmax)

    ax.set_xticks(range(len(pairs_order)))
    ax.set_xticklabels(["low–medium", "low–high", "medium–high"], fontsize=9)
    ax.set_yticks(range(len(emotions_order)))
    ax.set_yticklabels([e.capitalize() for e in emotions_order], fontsize=9)

    # annotate cells
    for i in range(len(emotions_order)):
        for j in range(len(pairs_order)):
            val = mat.values[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=7.5,
                    color="white" if val < (vmin + (vmax - vmin) * 0.55) else "black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine similarity", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title("Intensity-pair cosine similarity per emotion (Layer 13)", fontsize=10)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")


In [10]:

def plot_inter_emotion_cosine_heatmap(save_path: Path) -> None:
    """
    8×8 symmetric heatmap of inter-emotion cosine similarity (pooled CAA directions, Layer 13).
    Diagonal is masked. Colour scale spans [0, 1] to make the uniformly high values visually apparent.
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_cosine_similarity_L13.csv", index_col=0)
    emotions_order = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
    mat = df.reindex(index=emotions_order, columns=emotions_order).values.astype(float)

    # mask diagonal
    masked = np.ma.masked_where(np.eye(len(emotions_order), dtype=bool), mat)

    vmin, vmax = 0.0, 1.0

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad(color="#e8e8e8")  # diagonal colour
    im = ax.imshow(masked, aspect="equal", cmap=cmap, vmin=vmin, vmax=vmax)

    n = len(emotions_order)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    labels = [e.capitalize() for e in emotions_order]
    ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8.5)
    ax.set_yticklabels(labels, fontsize=8.5)

    # annotate off-diagonal only
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            val = mat[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=6.5,
                    color="white" if val > 0.6 else "black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine similarity", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title("Inter-emotion cosine similarity\nof pooled CAA directions (Layer 13)", fontsize=10)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")


In [11]:

def generate_intensity_consistency_table() -> str:
    """
    LaTeX table summarising intensity-pair cosine similarity at Layer 13.
    Columns: Pair | Mean | Min | Max (across 8 emotions).
    """
    df = pd.read_csv(CAA_GEOMETRY / "caa_intensity_consistency.csv")
    df13 = df[df["layer"] == 13]

    pairs_order = ["low-medium", "low-high", "medium-high"]
    rows = []
    for pair in pairs_order:
        subset = df13[df13["pair"] == pair]["cosine"]
        mean_v = subset.mean()
        min_v  = subset.min()
        max_v  = subset.max()
        label  = pair.replace("-", "–")  # en-dash for typography
        rows.append(
            f"  {label} & {mean_v:.3f} & {min_v:.3f} & {max_v:.3f} \\\\"
        )

    body = "\n".join(rows)
    table = r"""\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Intensity pair} &
  \textbf{Mean cosine} &
  \textbf{Min cosine} &
  \textbf{Max cosine} \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\caption{Summary of intensity-pair cosine similarity at Layer 13, computed across
         all eight emotion categories. Even for the largest intensity gap
         (\textit{low–high}), direction consistency remains above 0.98,
         indicating that intensity modulation primarily scales rather than
         redirects the CAA vector.}
\label{tab:intensity_consistency_L13}
\end{table}"""
    return table


In [12]:

# ── Generate CAA geometry figures & table ──────────────────────────────────────
plot_intensity_consistency_heatmap(FIGURES / "caa_intensity_consistency_heatmap_L13.pdf")
plot_inter_emotion_cosine_heatmap(FIGURES / "caa_inter_emotion_cosine_heatmap_L13.pdf")

tex_ic = generate_intensity_consistency_table()
print("\n── Intensity consistency LaTeX table ──────────────────────────────────")
print(tex_ic)
ic_path = FIGURES / "caa_intensity_consistency_table_L13.tex"
ic_path.write_text(tex_ic)
print(f"\nSaved: {ic_path}")


Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/caa_intensity_consistency_heatmap_L13.pdf
Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/caa_inter_emotion_cosine_heatmap_L13.pdf

── Intensity consistency LaTeX table ──────────────────────────────────
\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Intensity pair} &
  \textbf{Mean cosine} &
  \textbf{Min cosine} &
  \textbf{Max cosine} \\
\midrule
  low–medium & 0.996 & 0.995 & 0.998 \\
  low–high & 0.990 & 0.983 & 0.995 \\
  medium–high & 0.997 & 0.994 & 0.999 \\
\bottomrule
\end{tabular}
\caption{Summary of intensity-pair cosine similarity at Layer 13, computed across
         all eight emotion categories. Even for the largest intensity gap
         (\textit{low–high}), direction consistency remains above 0.98,
         indicating that intensity modulation primarily scales rather than
         redirects the CAA vector.}
\label{tab:intensity_consistency_L13}
\end{table}

Saved: /ho